# 05 - Seq2Seq and Attention Basics

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Describe the **encoder-decoder (seq2seq)** architecture and its components
- Explain the **fixed-size bottleneck** problem in basic seq2seq models
- Implement the **attention mechanism**: score functions, attention weights, and context vectors
- Build a simplified seq2seq model for a toy task (reverse a sequence)
- Add an attention layer and **visualize attention weights** as a heatmap
- Understand how attention leads to **Transformers** (DL600)

## Prerequisites

- Notebook 03: LSTM and GRU (using `nn.LSTM`/`nn.GRU`)
- Understanding of encoder and decoder concepts
- Matrix operations and softmax

## Table of Contents

1. [Sequence-to-Sequence Architecture](#1-sequence-to-sequence-architecture)
2. [The Fixed-Size Bottleneck Problem](#2-the-fixed-size-bottleneck-problem)
3. [Attention Mechanism](#3-attention-mechanism)
4. [Attention Equations](#4-attention-equations)
5. [Code: Seq2Seq for Sequence Reversal (No Attention)](#5-code-seq2seq-for-sequence-reversal-no-attention)
6. [Code: Adding Attention to the Decoder](#6-code-adding-attention-to-the-decoder)
7. [Code: Training and Evaluation](#7-code-training-and-evaluation)
8. [Code: Visualize Attention Weights](#8-code-visualize-attention-weights)
9. [Bridge to DL600: From Attention to Transformers](#9-bridge-to-dl600-from-attention-to-transformers)
10. [Exercise: Modify the Attention Scoring Function](#10-exercise-modify-the-attention-scoring-function)
11. [Common Mistakes & Debugging Tips](#11-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Sequence-to-Sequence Architecture

A **seq2seq** (sequence-to-sequence) model maps an input sequence to an output sequence of potentially different length. It consists of two parts:

### Encoder
- Processes the input sequence one token at a time
- Produces a set of hidden states $\bar{h}_1, \bar{h}_2, \ldots, \bar{h}_S$ (one per input token)
- The final hidden state is used as the **context vector** to initialize the decoder

### Decoder
- Generates the output sequence one token at a time
- At each step, takes the previous output token and the current hidden state
- Produces a probability distribution over the output vocabulary

### Applications

| Task | Input | Output |
|------|-------|--------|
| Machine translation | "The cat sat" | "Le chat assis" |
| Text summarization | Long article | Short summary |
| Chatbot | User message | Bot response |
| Code generation | Description | Code snippet |

In [ ]:
# Visualize the seq2seq architecture
fig, ax = plt.subplots(figsize=(14, 5))

# Encoder
enc_words = ['A', 'B', 'C', '<EOS>']
for i, w in enumerate(enc_words):
    # Input
    ax.add_patch(plt.Rectangle((i * 1.8, 0), 1.2, 0.5, facecolor='#AED6F1',
                                edgecolor='black', linewidth=1.2))
    ax.text(i * 1.8 + 0.6, 0.25, w, ha='center', va='center', fontsize=11)
    # Hidden state
    ax.add_patch(plt.Rectangle((i * 1.8, 1.0), 1.2, 0.5, facecolor='#F9E79F',
                                edgecolor='black', linewidth=1.2))
    ax.text(i * 1.8 + 0.6, 1.25, f'$\\bar{{h}}_{i+1}$', ha='center', va='center', fontsize=11)
    # Arrow up
    ax.annotate('', xy=(i * 1.8 + 0.6, 1.0), xytext=(i * 1.8 + 0.6, 0.5),
                arrowprops=dict(arrowstyle='->', color='black'))
    # Arrow right
    if i < len(enc_words) - 1:
        ax.annotate('', xy=((i + 1) * 1.8, 1.25), xytext=(i * 1.8 + 1.2, 1.25),
                    arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))

# Context vector arrow
ax.annotate('context\nvector', xy=(8.0, 1.25), xytext=(7.0, 1.25),
            arrowprops=dict(arrowstyle='->', color='#8E44AD', lw=2.5),
            fontsize=9, ha='center', va='center', color='#8E44AD')

# Decoder
dec_words = ['<SOS>', 'C', 'B', 'A']
dec_outputs = ['C', 'B', 'A', '<EOS>']
offset = 4.8

for i, (w_in, w_out) in enumerate(zip(dec_words, dec_outputs)):
    x_pos = offset + i * 1.8
    # Input
    ax.add_patch(plt.Rectangle((x_pos, 0), 1.2, 0.5, facecolor='#D5F5E3',
                                edgecolor='black', linewidth=1.2))
    ax.text(x_pos + 0.6, 0.25, w_in, ha='center', va='center', fontsize=11)
    # Hidden state
    ax.add_patch(plt.Rectangle((x_pos, 1.0), 1.2, 0.5, facecolor='#FADBD8',
                                edgecolor='black', linewidth=1.2))
    ax.text(x_pos + 0.6, 1.25, f'$h_{i+1}$', ha='center', va='center', fontsize=11)
    # Output
    ax.add_patch(plt.Rectangle((x_pos, 2.0), 1.2, 0.5, facecolor='#E8DAEF',
                                edgecolor='black', linewidth=1.2))
    ax.text(x_pos + 0.6, 2.25, w_out, ha='center', va='center', fontsize=11, fontweight='bold')
    # Arrows
    ax.annotate('', xy=(x_pos + 0.6, 1.0), xytext=(x_pos + 0.6, 0.5),
                arrowprops=dict(arrowstyle='->', color='black'))
    ax.annotate('', xy=(x_pos + 0.6, 2.0), xytext=(x_pos + 0.6, 1.5),
                arrowprops=dict(arrowstyle='->', color='black'))
    if i < len(dec_words) - 1:
        ax.annotate('', xy=(offset + (i + 1) * 1.8, 1.25),
                    xytext=(x_pos + 1.2, 1.25),
                    arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))

ax.text(2.5, 1.9, 'ENCODER', ha='center', fontsize=13, fontweight='bold', color='#2980B9')
ax.text(9.5, 2.8, 'DECODER', ha='center', fontsize=13, fontweight='bold', color='#C0392B')

ax.set_xlim(-0.5, 13)
ax.set_ylim(-0.3, 3.2)
ax.set_title('Seq2Seq: Encoder-Decoder Architecture (Reverse Sequence Task)', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 2. The Fixed-Size Bottleneck Problem

In the basic seq2seq model, the **entire input** must be compressed into a single fixed-size context vector (the encoder's final hidden state):

$$\text{context} = \bar{h}_S \in \mathbb{R}^d$$

### Problems

- **Information loss**: a 100-word sentence compressed to a 256-dim vector loses details
- **Long sequences suffer most**: the more tokens, the more information is lost
- **Early tokens are disadvantaged**: due to vanishing gradients in the encoder

### Empirical evidence

In machine translation, basic seq2seq performance **degrades rapidly** as sentence length increases beyond ~20 tokens.

### Solution: Attention

Instead of relying on a single context vector, let the decoder **attend to all encoder hidden states** at each decoding step.

---
## 3. Attention Mechanism

The attention mechanism (Bahdanau et al., 2014) gives the decoder **direct access** to all encoder hidden states.

At each decoder step $t$:

1. **Score** the decoder hidden state $h_t$ against each encoder hidden state $\bar{h}_s$
2. **Normalize** scores into attention weights via softmax
3. **Compute** a weighted sum (context vector) of encoder states
4. **Concatenate** context with decoder hidden state and predict

### Intuition

When translating "The cat sat on the mat" to French:
- When generating "chat" (cat), the decoder should focus on the word "cat" in the input
- When generating "tapis" (mat), the decoder should focus on "mat"
- Attention learns this alignment automatically

---
## 4. Attention Equations

### Score functions

The score measures how relevant encoder state $\bar{h}_s$ is to decoder state $h_t$:

| Name | Equation | Notes |
|------|----------|-------|
| **Dot product** | $\text{score}(h_t, \bar{h}_s) = h_t^\top \bar{h}_s$ | Simplest, requires same dimensions |
| **General** (Luong) | $\text{score}(h_t, \bar{h}_s) = h_t^\top W_a \bar{h}_s$ | Learnable weight matrix |
| **Additive** (Bahdanau) | $\text{score}(h_t, \bar{h}_s) = v_a^\top \tanh(W_1 h_t + W_2 \bar{h}_s)$ | Most flexible, more parameters |

### Attention weights

$$\alpha_{ts} = \frac{\exp(\text{score}(h_t, \bar{h}_s))}{\sum_{s'=1}^{S} \exp(\text{score}(h_t, \bar{h}_{s'}))}$$

The weights $\alpha_{ts}$ sum to 1 across all encoder positions $s$ for a given decoder step $t$.

### Context vector

$$c_t = \sum_{s=1}^{S} \alpha_{ts} \bar{h}_s$$

The context vector $c_t$ is a weighted average of encoder hidden states, with weights determined by relevance to the current decoder step.

---
## 5. Code: Seq2Seq for Sequence Reversal (No Attention)

We start with a basic seq2seq model (no attention) for a toy task: **reverse a sequence of integers**.

Example: `[3, 7, 1, 5]` -> `[5, 1, 7, 3]`

In [ ]:
set_global_seed(42)

# Task setup
# Vocabulary: 0=PAD, 1=SOS (start of sequence), 2=EOS (end of sequence), 3..12=digits
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
VOCAB_SIZE = 13  # 0-12 (PAD, SOS, EOS, + 10 digit tokens 3..12)

def generate_reverse_data(n_samples, min_len=3, max_len=8):
    """Generate (input, target) pairs for the sequence reversal task.
    
    Input:  [d1, d2, ..., dn, EOS, PAD, PAD, ...]
    Target: [SOS, dn, ..., d2, d1, EOS, PAD, ...]
    """
    max_total_len = max_len + 2  # +1 for EOS in input, +1 for SOS/EOS in target
    inputs = []
    targets = []
    
    for _ in range(n_samples):
        seq_len = np.random.randint(min_len, max_len + 1)
        # Random digits (tokens 3..12)
        seq = np.random.randint(3, VOCAB_SIZE, size=seq_len).tolist()
        
        # Input: seq + EOS + padding
        inp = seq + [EOS_IDX] + [PAD_IDX] * (max_total_len - seq_len - 1)
        # Target: SOS + reversed_seq + EOS + padding
        tgt = [SOS_IDX] + seq[::-1] + [EOS_IDX] + [PAD_IDX] * (max_total_len - seq_len - 2)
        
        inputs.append(inp)
        targets.append(tgt)
    
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

X_train, y_train = generate_reverse_data(3000)
X_test, y_test = generate_reverse_data(500)

print(f"Training: X={X_train.shape}, y={y_train.shape}")
print(f"Test:     X={X_test.shape}, y={y_test.shape}")
print(f"\nExample:")
print(f"  Input:  {X_train[0].tolist()}")
print(f"  Target: {y_train[0].tolist()}")

In [ ]:
class Encoder(nn.Module):
    """LSTM encoder: embeds input tokens and processes them sequentially."""
    
    def __init__(self, vocab_size, embed_dim, hidden_size, n_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers > 1 else 0)
    
    def forward(self, x):
        """Encode input sequence.
        
        Args:
            x: (batch, src_len) input token indices
        Returns:
            encoder_outputs: (batch, src_len, hidden_size) hidden states at all steps
            (h_n, c_n): final hidden and cell states
        """
        embedded = self.embedding(x)  # (batch, src_len, embed_dim)
        encoder_outputs, (h_n, c_n) = self.lstm(embedded)
        return encoder_outputs, (h_n, c_n)


class DecoderNoAttention(nn.Module):
    """LSTM decoder without attention: uses only the context vector."""
    
    def __init__(self, vocab_size, embed_dim, hidden_size, n_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden):
        """Decode one step.
        
        Args:
            x: (batch, 1) input token at current step
            hidden: (h, c) from previous step
        Returns:
            output: (batch, 1, vocab_size) logits
            hidden: updated (h, c)
        """
        embedded = self.embedding(x)        # (batch, 1, embed_dim)
        output, hidden = self.lstm(embedded, hidden)
        output = self.fc(output)             # (batch, 1, vocab_size)
        return output, hidden


class Seq2SeqNoAttention(nn.Module):
    """Complete seq2seq model without attention."""
    
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """Forward pass with optional teacher forcing.
        
        Args:
            src: (batch, src_len) input sequences
            tgt: (batch, tgt_len) target sequences (for teacher forcing)
            teacher_forcing_ratio: probability of using ground truth as next input
        Returns:
            outputs: (batch, tgt_len-1, vocab_size) predictions for each step
        """
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        
        # Encode
        _, hidden = self.encoder(src)
        
        # Decode step by step
        outputs = []
        input_tok = tgt[:, 0:1]  # SOS token
        
        for t in range(1, tgt_len):
            output, hidden = self.decoder(input_tok, hidden)
            outputs.append(output)
            
            # Teacher forcing: use ground truth or predicted token
            if np.random.random() < teacher_forcing_ratio:
                input_tok = tgt[:, t:t+1]  # ground truth
            else:
                input_tok = output.argmax(dim=-1)  # predicted
        
        outputs = torch.cat(outputs, dim=1)  # (batch, tgt_len-1, vocab_size)
        return outputs


# Build model
set_global_seed(42)

EMBED_DIM = 32
HIDDEN_SIZE = 64

encoder_na = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
decoder_na = DecoderNoAttention(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
model_na = Seq2SeqNoAttention(encoder_na, decoder_na)

n_params = sum(p.numel() for p in model_na.parameters())
print(f"Seq2Seq (no attention) parameters: {n_params:,}")
print(model_na)

---
## 6. Code: Adding Attention to the Decoder

Now we add a **dot-product attention** layer to the decoder.

In [ ]:
class Attention(nn.Module):
    """Dot-product attention module."""
    
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
    
    def forward(self, decoder_hidden, encoder_outputs):
        """Compute attention weights and context vector.
        
        Args:
            decoder_hidden: (batch, 1, hidden_size) current decoder hidden state
            encoder_outputs: (batch, src_len, hidden_size) all encoder hidden states
        
        Returns:
            context: (batch, 1, hidden_size) weighted sum of encoder states
            attn_weights: (batch, 1, src_len) attention weights
        """
        # Dot-product scores: (batch, 1, hidden) @ (batch, hidden, src_len) = (batch, 1, src_len)
        scores = torch.bmm(decoder_hidden, encoder_outputs.transpose(1, 2))
        
        # Normalize to attention weights
        attn_weights = F.softmax(scores, dim=-1)  # (batch, 1, src_len)
        
        # Context vector: weighted sum of encoder outputs
        context = torch.bmm(attn_weights, encoder_outputs)  # (batch, 1, hidden_size)
        
        return context, attn_weights


class DecoderWithAttention(nn.Module):
    """LSTM decoder with attention over encoder outputs."""
    
    def __init__(self, vocab_size, embed_dim, hidden_size, n_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.attention = Attention(hidden_size)
        # Output layer takes concatenation of LSTM output and context
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
    
    def forward(self, x, hidden, encoder_outputs):
        """Decode one step with attention.
        
        Args:
            x: (batch, 1) input token
            hidden: (h, c) from previous step
            encoder_outputs: (batch, src_len, hidden_size)
        Returns:
            output: (batch, 1, vocab_size) logits
            hidden: updated (h, c)
            attn_weights: (batch, 1, src_len)
        """
        embedded = self.embedding(x)  # (batch, 1, embed_dim)
        lstm_out, hidden = self.lstm(embedded, hidden)  # (batch, 1, hidden_size)
        
        # Attention
        context, attn_weights = self.attention(lstm_out, encoder_outputs)
        
        # Concatenate LSTM output and context, then project
        combined = torch.cat([lstm_out, context], dim=-1)  # (batch, 1, hidden_size*2)
        output = self.fc(combined)  # (batch, 1, vocab_size)
        
        return output, hidden, attn_weights


class Seq2SeqWithAttention(nn.Module):
    """Complete seq2seq model with attention."""
    
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        
        # Encode
        encoder_outputs, hidden = self.encoder(src)
        
        # Decode step by step
        outputs = []
        attentions = []
        input_tok = tgt[:, 0:1]  # SOS token
        
        for t in range(1, tgt_len):
            output, hidden, attn_w = self.decoder(input_tok, hidden, encoder_outputs)
            outputs.append(output)
            attentions.append(attn_w)
            
            if np.random.random() < teacher_forcing_ratio:
                input_tok = tgt[:, t:t+1]
            else:
                input_tok = output.argmax(dim=-1)
        
        outputs = torch.cat(outputs, dim=1)
        attentions = torch.cat(attentions, dim=1)  # (batch, tgt_len-1, src_len)
        return outputs, attentions


# Build attention model
set_global_seed(42)

encoder_attn = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
decoder_attn = DecoderWithAttention(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
model_attn = Seq2SeqWithAttention(encoder_attn, decoder_attn)

n_params_attn = sum(p.numel() for p in model_attn.parameters())
print(f"Seq2Seq (with attention) parameters: {n_params_attn:,}")
print(f"Extra parameters from attention: {n_params_attn - n_params:,}")

---
## 7. Code: Training and Evaluation

We train both models and compare their performance.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
N_EPOCHS = 30
LR = 0.001

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


def train_seq2seq(model, train_loader, n_epochs, lr, has_attention=False):
    """Train a seq2seq model."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    losses = []
    
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        
        # Decrease teacher forcing over time
        tf_ratio = max(0.1, 1.0 - epoch / n_epochs)
        
        for src, tgt in train_loader:
            optimizer.zero_grad()
            
            if has_attention:
                output, _ = model(src, tgt, teacher_forcing_ratio=tf_ratio)
            else:
                output = model(src, tgt, teacher_forcing_ratio=tf_ratio)
            
            # output: (batch, tgt_len-1, vocab_size)
            # tgt[:, 1:]: (batch, tgt_len-1) -- skip SOS
            output_flat = output.reshape(-1, VOCAB_SIZE)
            tgt_flat = tgt[:, 1:].reshape(-1)
            
            loss = loss_fn(output_flat, tgt_flat)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:2d}/{n_epochs} | Loss: {avg_loss:.4f} | TF ratio: {tf_ratio:.2f}")
    
    return losses


def evaluate_seq2seq(model, test_loader, has_attention=False):
    """Evaluate seq2seq model: compute sequence-level accuracy."""
    model.eval()
    correct_seqs = 0
    total_seqs = 0
    correct_tokens = 0
    total_tokens = 0
    
    with torch.no_grad():
        for src, tgt in test_loader:
            if has_attention:
                output, _ = model(src, tgt, teacher_forcing_ratio=0.0)  # no TF at eval
            else:
                output = model(src, tgt, teacher_forcing_ratio=0.0)
            
            preds = output.argmax(dim=-1)  # (batch, tgt_len-1)
            targets = tgt[:, 1:]           # skip SOS
            
            # Mask padding
            mask = targets != PAD_IDX
            correct_tokens += ((preds == targets) & mask).sum().item()
            total_tokens += mask.sum().item()
            
            # Sequence-level: all non-padding tokens must match
            for i in range(src.size(0)):
                m = mask[i]
                if (preds[i][m] == targets[i][m]).all():
                    correct_seqs += 1
                total_seqs += 1
    
    token_acc = correct_tokens / max(total_tokens, 1)
    seq_acc = correct_seqs / max(total_seqs, 1)
    return token_acc, seq_acc

print("Setup complete. Ready to train.")

In [ ]:
# Train model WITHOUT attention
set_global_seed(42)
encoder_na = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
decoder_na = DecoderNoAttention(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
model_na = Seq2SeqNoAttention(encoder_na, decoder_na)

print("Training Seq2Seq WITHOUT attention:")
losses_na = train_seq2seq(model_na, train_loader, N_EPOCHS, LR, has_attention=False)

tok_acc_na, seq_acc_na = evaluate_seq2seq(model_na, test_loader, has_attention=False)
print(f"\nNo-attention | Token accuracy: {tok_acc_na:.3f} | Sequence accuracy: {seq_acc_na:.3f}")

In [ ]:
# Train model WITH attention
set_global_seed(42)
encoder_attn = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
decoder_attn = DecoderWithAttention(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
model_attn = Seq2SeqWithAttention(encoder_attn, decoder_attn)

print("Training Seq2Seq WITH attention:")
losses_attn = train_seq2seq(model_attn, train_loader, N_EPOCHS, LR, has_attention=True)

tok_acc_attn, seq_acc_attn = evaluate_seq2seq(model_attn, test_loader, has_attention=True)
print(f"\nWith attention | Token accuracy: {tok_acc_attn:.3f} | Sequence accuracy: {seq_acc_attn:.3f}")

In [ ]:
# Compare training losses
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_na, 'r-', linewidth=1.5, label='No Attention')
axes[0].plot(losses_attn, 'b-', linewidth=1.5, label='With Attention')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss Comparison', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bar chart of accuracies
labels = ['Token Accuracy', 'Sequence Accuracy']
na_vals = [tok_acc_na, seq_acc_na]
attn_vals = [tok_acc_attn, seq_acc_attn]

x_pos = np.arange(len(labels))
width = 0.3

axes[1].bar(x_pos - width/2, na_vals, width, color='#e74c3c', label='No Attention', alpha=0.8)
axes[1].bar(x_pos + width/2, attn_vals, width, color='#3498db', label='With Attention', alpha=0.8)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy Comparison', fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(labels)
axes[1].legend()
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3, axis='y')

# Annotate bars
for i, (v1, v2) in enumerate(zip(na_vals, attn_vals)):
    axes[1].text(i - width/2, v1 + 0.02, f'{v1:.2f}', ha='center', fontsize=10)
    axes[1].text(i + width/2, v2 + 0.02, f'{v2:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## 8. Code: Visualize Attention Weights

The attention weights $\alpha_{ts}$ tell us which encoder positions the decoder focuses on at each output step. We visualize this as a heatmap.

In [ ]:
def visualize_attention(model, src_seq, tgt_seq):
    """Visualize attention weights for a single example."""
    model.eval()
    with torch.no_grad():
        src = src_seq.unsqueeze(0)  # (1, src_len)
        tgt = tgt_seq.unsqueeze(0)  # (1, tgt_len)
        
        # Forward pass with no teacher forcing
        output, attentions = model(src, tgt, teacher_forcing_ratio=0.0)
        # attentions: (1, tgt_len-1, src_len)
    
    attn_matrix = attentions.squeeze(0).numpy()  # (tgt_len-1, src_len)
    preds = output.argmax(dim=-1).squeeze(0).numpy()  # (tgt_len-1,)
    
    # Get token strings
    token_map = {0: 'PAD', 1: 'SOS', 2: 'EOS'}
    for i in range(3, VOCAB_SIZE):
        token_map[i] = str(i - 3)  # map 3->"0", 4->"1", etc.
    
    src_tokens = [token_map[t.item()] for t in src_seq]
    tgt_tokens = [token_map[t.item()] for t in tgt_seq[1:]]  # skip SOS
    pred_tokens = [token_map[int(p)] for p in preds]
    
    # Find non-PAD length
    src_len = (src_seq != PAD_IDX).sum().item()
    tgt_len = (tgt_seq[1:] != PAD_IDX).sum().item()
    
    # Crop to non-PAD region
    attn_crop = attn_matrix[:tgt_len, :src_len]
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(attn_crop, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    
    ax.set_xticks(range(src_len))
    ax.set_xticklabels(src_tokens[:src_len], fontsize=12)
    ax.set_yticks(range(tgt_len))
    ax.set_yticklabels([f'{tgt_tokens[i]} (pred: {pred_tokens[i]})' for i in range(tgt_len)],
                       fontsize=11)
    
    ax.set_xlabel('Encoder Input', fontsize=12)
    ax.set_ylabel('Decoder Output', fontsize=12)
    ax.set_title('Attention Weights Heatmap', fontweight='bold', fontsize=13)
    
    # Annotate cells with weight values
    for i in range(tgt_len):
        for j in range(src_len):
            val = attn_crop[i, j]
            color = 'white' if val > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=9, color=color)
    
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Attention Weight')
    plt.tight_layout()
    plt.show()
    
    return attn_crop

In [ ]:
# Visualize attention for several test examples
for example_idx in [0, 5, 10]:
    src_seq = X_test[example_idx]
    tgt_seq = y_test[example_idx]
    
    token_map = {0: 'PAD', 1: 'SOS', 2: 'EOS'}
    for i in range(3, VOCAB_SIZE):
        token_map[i] = str(i - 3)
    
    src_toks = [token_map[t.item()] for t in src_seq if t.item() != PAD_IDX]
    tgt_toks = [token_map[t.item()] for t in tgt_seq if t.item() != PAD_IDX]
    print(f"Example {example_idx}: Input={src_toks}, Expected Output={tgt_toks}")
    
    attn = visualize_attention(model_attn, src_seq, tgt_seq)
    print()

For the **reversal task**, we expect attention to show a clear **anti-diagonal** pattern: the first output token attends to the last input token, the second output attends to the second-to-last, and so on.

---
## 9. Bridge to DL600: From Attention to Transformers

The attention mechanism we built here is the foundation of the **Transformer** architecture (Vaswani et al., 2017):

### From RNN+Attention to Transformers

| Feature | RNN + Attention | Transformer |
|---------|----------------|-------------|
| Sequence processing | Sequential (one token at a time) | Parallel (all tokens at once) |
| Attention | Decoder attends to encoder | **Self-attention**: every token attends to every other |
| Position info | Built into RNN recurrence | Added via **positional encoding** |
| Training speed | Slow (sequential) | Fast (parallelizable) |
| Long-range deps | Better with attention, still limited | Excellent (direct connections) |

### Key Transformer innovations

1. **Self-attention**: tokens attend to all other tokens in the **same** sequence (not just encoder-to-decoder)
2. **Multi-head attention**: multiple attention mechanisms in parallel, each learning different relationships
3. **Scaled dot-product**: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$
4. **No recurrence**: eliminates sequential dependency, enabling massive parallelism

The DL600 module covers Transformers in depth.

---
## 10. Exercise: Modify the Attention Scoring Function

**Task:**

1. Implement a **General (Luong)** attention scoring function: $\text{score}(h_t, \bar{h}_s) = h_t^\top W_a \bar{h}_s$
2. Replace the dot-product attention in the model with this new scoring function
3. Train the model and compare results with the dot-product attention
4. Which scoring function performs better on this task? Why might that be?

```python
# Your code here

class GeneralAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # TODO: add learnable weight matrix W_a
        # self.W_a = nn.Linear(hidden_size, hidden_size, bias=False)
    
    def forward(self, decoder_hidden, encoder_outputs):
        # TODO: compute scores using W_a
        # transformed = self.W_a(encoder_outputs)  # (batch, src_len, hidden)
        # scores = torch.bmm(decoder_hidden, transformed.transpose(1, 2))
        # attn_weights = F.softmax(scores, dim=-1)
        # context = torch.bmm(attn_weights, encoder_outputs)
        # return context, attn_weights
        pass
```

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution

In [ ]:
# ----- Solution -----

class GeneralAttention(nn.Module):
    """General (Luong) attention: score = h_t^T W_a h_s"""
    
    def __init__(self, hidden_size):
        super().__init__()
        self.W_a = nn.Linear(hidden_size, hidden_size, bias=False)
    
    def forward(self, decoder_hidden, encoder_outputs):
        # Transform encoder outputs: (batch, src_len, hidden) -> (batch, src_len, hidden)
        transformed = self.W_a(encoder_outputs)
        # Score: (batch, 1, hidden) @ (batch, hidden, src_len) = (batch, 1, src_len)
        scores = torch.bmm(decoder_hidden, transformed.transpose(1, 2))
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights, encoder_outputs)
        return context, attn_weights


class DecoderWithGeneralAttention(nn.Module):
    """Decoder using General (Luong) attention."""
    
    def __init__(self, vocab_size, embed_dim, hidden_size, n_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=n_layers,
                            batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.attention = GeneralAttention(hidden_size)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
    
    def forward(self, x, hidden, encoder_outputs):
        embedded = self.embedding(x)
        lstm_out, hidden = self.lstm(embedded, hidden)
        context, attn_weights = self.attention(lstm_out, encoder_outputs)
        combined = torch.cat([lstm_out, context], dim=-1)
        output = self.fc(combined)
        return output, hidden, attn_weights


# Train with General attention
set_global_seed(42)
encoder_gen = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
decoder_gen = DecoderWithGeneralAttention(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE)
model_gen = Seq2SeqWithAttention(encoder_gen, decoder_gen)

print("Training Seq2Seq with GENERAL attention:")
losses_gen = train_seq2seq(model_gen, train_loader, N_EPOCHS, LR, has_attention=True)

tok_acc_gen, seq_acc_gen = evaluate_seq2seq(model_gen, test_loader, has_attention=True)
print(f"\nGeneral attention | Token acc: {tok_acc_gen:.3f} | Sequence acc: {seq_acc_gen:.3f}")
print(f"Dot-product attn  | Token acc: {tok_acc_attn:.3f} | Sequence acc: {seq_acc_attn:.3f}")

In [ ]:
# Compare all three models
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(losses_na, 'r-', linewidth=1.5, label='No Attention')
ax.plot(losses_attn, 'b-', linewidth=1.5, label='Dot-Product Attention')
ax.plot(losses_gen, 'g-', linewidth=1.5, label='General (Luong) Attention')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Seq2Seq Training Loss: Three Variants', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal test results:")
print(f"  No attention:         token_acc={tok_acc_na:.3f}  seq_acc={seq_acc_na:.3f}")
print(f"  Dot-product attention: token_acc={tok_acc_attn:.3f}  seq_acc={seq_acc_attn:.3f}")
print(f"  General attention:     token_acc={tok_acc_gen:.3f}  seq_acc={seq_acc_gen:.3f}")
print(f"\nNote: On this simple task, both attention types should outperform no attention.")
print(f"The General attention has a learnable W_a matrix that adds flexibility.")

---
## 11. Common Mistakes & Debugging Tips

**1. Forgetting to pass `<SOS>` as the first decoder input**
- The decoder needs a start token to begin generation. Always feed `SOS` at step 0.
- Forgetting this shifts all predictions by one position.

**2. Teacher forcing ratio too high during evaluation**
- During training, teacher forcing (feeding ground truth) speeds learning.
- During evaluation, set `teacher_forcing_ratio=0.0` -- the model must use its own predictions.
- A common bug: leaving teacher forcing on during evaluation gives inflated accuracy.

**3. Not ignoring PAD tokens in the loss**
- Use `nn.CrossEntropyLoss(ignore_index=PAD_IDX)` so padding does not contribute to the loss.
- Without this, the model wastes capacity learning to predict PAD.

**4. Attention dimension mismatch**
- For dot-product attention, encoder and decoder hidden sizes must match.
- For General/Additive attention, the weight matrix handles dimension differences.

**5. Not decreasing teacher forcing over time**
- Start with high teacher forcing (0.8--1.0) and gradually decrease to (0.1--0.2).
- This helps the model learn from its own predictions, reducing the train-test gap.

**6. Confusing `output` shapes in seq2seq**
- Encoder output: `(batch, src_len, hidden_size)` -- all encoder hidden states
- Decoder output: `(batch, 1, vocab_size)` at each step -- logits over vocabulary
- Attention weights: `(batch, 1, src_len)` -- one weight per encoder position

**7. Gradient explosion in seq2seq**
- Always clip gradients with `clip_grad_norm_`. Seq2seq models are particularly prone to gradient spikes.
- If loss becomes `NaN`, reduce learning rate and increase gradient clipping.

---

**Next module:** [DL600 - Transformers and Self-Attention](../../notebooks/DL600_Transformers/) -- we remove recurrence entirely and build models based purely on attention, enabling massive parallelism and state-of-the-art NLP performance.